# Más allá de la tabla · Notebook 3: análisis exploratorio

**Pregunta guía:** ¿Cómo se relacionan los tiros, tiros al arco, córners, faltas y tarjetas con los goles y los puntos obtenidos por los equipos de la Premier League, y qué diferencias existen entre su rendimiento como local y visitante?

**Entradas** (las genera el notebook 2):

- `estadisticas_equipo_temporada.csv`: una fila por equipo, temporada y condición (Total, Local o Visitante).
- `equipo_partido.csv`: una fila por equipo y partido. Solo se usa para los resultados por temporada y el gráfico de puntos acumulados.

**Unidad de análisis: equipo-temporada.** Cada observación es, por ejemplo, "Arsenal en 2023/24". Hay 100 (20 equipos × 5 temporadas) por cada condición.

**Usamos promedios por partido**, no totales. Así se pueden comparar los 19 partidos de local con los 19 de visitante, y el West Ham 2021/22, al que le falta el dato de tiros de un partido, no queda en desventaja.

**Contenido**

1. Inspección inicial
2. Estadísticas descriptivas con NumPy
3. Comparación entre temporadas
4. Local frente a visitante
5. Correlaciones
6. Gráficos exploratorios
7. Hallazgos y limitaciones
8. Guardar los resultados

## Paso 0. Cargar los datos y preparar los gráficos

Suban `estadisticas_equipo_temporada.csv` y `equipo_partido.csv` al panel de archivos de Colab.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for archivo in ["estadisticas_equipo_temporada.csv", "equipo_partido.csv"]:
    if not Path(archivo).exists():
        raise FileNotFoundError(f"Suban {archivo} (lo genera el notebook 2).")

estadisticas = pd.read_csv("estadisticas_equipo_temporada.csv")
equipo_partido = pd.read_csv("equipo_partido.csv", parse_dates=["Fecha"])

assert estadisticas.shape == (300, 27), "No es el archivo del notebook 2"
assert equipo_partido.shape == (3800, 21), "No es el archivo del notebook 2"

# Una tabla por condición: cada una tiene 100 filas (equipo-temporada).
total = estadisticas[estadisticas["Condicion"] == "Total"].reset_index(drop=True)
local = estadisticas[estadisticas["Condicion"] == "Local"].reset_index(drop=True)
visitante = estadisticas[estadisticas["Condicion"] == "Visitante"].reset_index(drop=True)

print(f"Total: {len(total)} · Local: {len(local)} · Visitante: {len(visitante)} observaciones")

In [ ]:
# Diccionario: columna -> nombre legible. Lo usan las tablas y los gráficos.
VARIABLES_JUEGO = {
    "TirosPorPartido": "Tiros",
    "TirosArcoPorPartido": "Tiros al arco",
    "CornersPorPartido": "Córners",
    "FaltasPorPartido": "Faltas",
    "AmarillasPorPartido": "Amarillas",
    "RojasPorPartido": "Rojas",
}
VARIABLES_RESULTADO = {
    "GolesFavorPorPartido": "Goles a favor",
    "GolesContraPorPartido": "Goles en contra",
    "PuntosPorPartido": "Puntos",
}
TODAS = {**VARIABLES_JUEGO, **VARIABLES_RESULTADO}

# Colores fijos: local siempre azul y visitante siempre naranja.
# Paleta comprobada para personas con daltonismo.
COLOR_LOCAL = "#2a78d6"
COLOR_VISITANTE = "#eb6834"
COLOR_TERCERO = "#1baf7a"
COLOR_TEXTO = "#52514e"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#bdbcb6",
    "axes.labelcolor": COLOR_TEXTO,
    "xtick.color": COLOR_TEXTO,
    "ytick.color": COLOR_TEXTO,
    "axes.grid": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "lines.linewidth": 2,
})

## 1. Inspección inicial

Miramos la tabla de estadísticas por equipo y temporada con las tres funciones de exploración de pandas.

In [ ]:
total[["Temporada", "Equipo", "PJ", "Puntos", "GolesFavor", "GolesContra"] + list(VARIABLES_JUEGO)].head()

In [ ]:
total.info()

In [ ]:
total[list(TODAS)].describe().round(2)

`describe()` confirma que las 100 observaciones tienen dato en todas las variables (`count` = 100): no hay valores faltantes en los promedios por partido. El partido sin tiros del West Ham ya se tuvo en cuenta en el notebook 2, al dividir sus tiros entre los partidos con dato.

## 2. Estadísticas descriptivas con NumPy

Calculamos cinco estadísticas para cada variable. Cada una aporta algo distinto:

| Estadística | Qué dice | Función de NumPy |
|---|---|---|
| Media | El valor típico de un equipo | `np.mean` |
| Mediana | El valor del medio. Si se aleja mucho de la media, unos pocos equipos extremos la están arrastrando | `np.median` |
| Desviación estándar | Cuánto se diferencian los equipos entre sí | `np.std(..., ddof=1)` |
| Mínimo y máximo | Los casos extremos, para saber qué equipos son | `np.min`, `np.max` |

**Sobre `ddof=1`:** por defecto `np.std` divide entre *n*; con `ddof=1` divide entre *n − 1*, que es la desviación estándar de una muestra. Es la que usa pandas en `describe()`, así que con `ddof=1` los dos resultados coinciden.

In [ ]:
def estadisticas_descriptivas(valores):
    """Calcula cinco estadísticas descriptivas de una variable con NumPy.

    Args:
        valores: serie o arreglo de números, sin valores faltantes.

    Returns:
        Diccionario con n, media, mediana, desviación estándar, mínimo y máximo.
    """
    arreglo = np.asarray(valores, dtype=float)
    if np.isnan(arreglo).any():
        raise ValueError("Hay valores faltantes: hay que tratarlos antes de calcular.")
    return {
        "n": len(arreglo),
        "Media": np.mean(arreglo),
        "Mediana": np.median(arreglo),
        "Desv. estándar": np.std(arreglo, ddof=1),
        "Mínimo": np.min(arreglo),
        "Máximo": np.max(arreglo),
    }


filas = []
for columna, nombre in TODAS.items():
    fila = {"Variable": nombre}
    fila.update(estadisticas_descriptivas(total[columna]))
    filas.append(fila)

descriptivas = pd.DataFrame(filas).set_index("Variable")
descriptivas.round(2)

Comprobamos que NumPy y pandas dan los mismos resultados. Si no coincidieran, habría un error en nuestra función.

In [ ]:
resumen_pandas = total[list(TODAS)].describe()
for columna, nombre in TODAS.items():
    assert np.isclose(descriptivas.loc[nombre, "Media"], resumen_pandas.loc["mean", columna])
    assert np.isclose(descriptivas.loc[nombre, "Desv. estándar"], resumen_pandas.loc["std", columna])
    assert np.isclose(descriptivas.loc[nombre, "Mediana"], resumen_pandas.loc["50%", columna])
print("NumPy y pandas coinciden en media, mediana y desviación estándar.")

¿Quiénes son los casos extremos? `idxmax()` e `idxmin()` devuelven la fila del máximo y del mínimo.

In [ ]:
def caso_extremo(tabla, columna, buscar_maximo=True):
    """Devuelve el texto "Equipo Temporada (valor)" del máximo o el mínimo de una columna."""
    if buscar_maximo:
        fila = tabla.loc[tabla[columna].idxmax()]
    else:
        fila = tabla.loc[tabla[columna].idxmin()]
    return f"{fila['Equipo']} {fila['Temporada']} ({fila[columna]:.2f})"


for columna, nombre in TODAS.items():
    print(f"{nombre:16} máximo: {caso_extremo(total, columna):32} "
          f"mínimo: {caso_extremo(total, columna, buscar_maximo=False)}")

**Cómo leer estos resultados** (cifras por partido, con los datos auditados):

- Un equipo típico suma **1,39 puntos**, hace **12,9 tiros** y **4,5 tiros al arco**, y saca **5,2 córners** por partido.
- **Los puntos son la variable con más diferencias entre equipos.** Van de 0,32 (Southampton 2024/25) a 2,45 (Man City 2021/22): el mejor equipo suma casi ocho veces lo que el peor.
- **Las tarjetas rojas son raras:** la media es 0,06 por partido, es decir, poco más de 2 por temporada. Con tan pocos casos, cualquier relación con los puntos será poco fiable.
- En los tiros, la media (12,9) es algo mayor que la mediana (12,5): unos pocos equipos que tiran muchísimo, como el Liverpool 2023/24 (20,8 tiros por partido), suben la media.

## 3. Comparación entre temporadas

Calculamos la media de los equipos en cada temporada. Si alguna temporada fuera muy distinta, habría que tenerlo en cuenta al juntarlas.

In [ ]:
por_temporada = total.groupby("Temporada")[list(TODAS)].mean().rename(columns=TODAS)
por_temporada.round(2)

Los promedios de juego son parecidos entre temporadas. La que más se aparta es 2023/24, con más goles (1,64 por partido y equipo, frente a 1,35 en 2020/21).

Ahora miramos los **resultados de los partidos** en cada temporada. Esta tabla está en otro nivel: cuenta **partidos**, no equipos. Usamos las filas del equipo local de `equipo_partido`: si el local ganó, es una victoria local.

In [ ]:
partidos_local = equipo_partido[equipo_partido["Condicion"] == "Local"]
resultados = pd.crosstab(partidos_local["Temporada"], partidos_local["Resultado"])
resultados = resultados[["V", "E", "D"]]
resultados.columns = ["Gana el local", "Empate", "Gana el visitante"]

porcentajes = (resultados.div(resultados.sum(axis=1), axis=0) * 100).round(1)
print("Número de partidos:")
print(resultados)
print("\nPorcentaje de partidos:")
print(porcentajes)

**2020/21 es la única temporada en que los visitantes ganaron más partidos que los locales** (153 frente a 144). Esa temporada se jugó casi entera sin público por la pandemia: los aficionados solo volvieron, con aforo limitado, en un breve periodo de diciembre de 2020 y en las dos últimas jornadas de mayo de 2021 (Wikipedia, *2020–21 Premier League*; Cyprus Mail, 5 de mayo de 2021).

Esto **coincide** con la ausencia de público, pero no lo demuestra: con estos datos no podemos separar el efecto del público de otros cambios de esa temporada, como el calendario apretado o la falta de pretemporada. Lo retomamos en la sección siguiente.

## 4. Local frente a visitante

Para cada equipo y temporada restamos su promedio como visitante a su promedio como local. Es una **comparación pareada**: comparamos a cada equipo consigo mismo, así la diferencia no depende de si el equipo es bueno o malo.

- Diferencia **positiva**: el equipo tiene más de esa estadística de local.
- Diferencia **negativa**: tiene más de visitante.

In [ ]:
indice = ["Temporada", "Equipo"]
de_local = local.set_index(indice)[list(TODAS)]
de_visitante = visitante.set_index(indice)[list(TODAS)]
diferencia = de_local - de_visitante  # pandas alinea las filas por temporada y equipo

comparacion = pd.DataFrame({
    "Local": de_local.mean(),
    "Visitante": de_visitante.mean(),
    "Diferencia (L − V)": diferencia.mean(),
}).rename(index=TODAS)
comparacion.round(2)

In [ ]:
mas_puntos_local = int((diferencia["PuntosPorPartido"] > 0).sum())
iguales = int((diferencia["PuntosPorPartido"] == 0).sum())
mas_puntos_visitante = int((diferencia["PuntosPorPartido"] < 0).sum())

print(f"De {len(diferencia)} equipo-temporada:")
print(f"  {mas_puntos_local} sumaron más puntos por partido de local")
print(f"  {mas_puntos_visitante} sumaron más de visitante")
print(f"  {iguales} sumaron lo mismo")

**Lo que muestran los datos:**

- De local, los equipos suman de media **0,28 puntos más por partido**, marcan 0,21 goles más y reciben 0,21 menos.
- También **tiran más** (+2,2 tiros y +0,7 tiros al arco), sacan **un córner más** y cometen **algo menos de faltas** (−0,2) y de amarillas (−0,2). En rojas no hay diferencia.
- **73 de los 100** equipo-temporada sumaron más puntos de local que de visitante.

Veamos si la ventaja de local es igual en todas las temporadas.

In [ ]:
por_temporada_lv = pd.DataFrame({
    "Puntos local": de_local.groupby("Temporada")["PuntosPorPartido"].mean(),
    "Puntos visitante": de_visitante.groupby("Temporada")["PuntosPorPartido"].mean(),
    "Ventaja local": diferencia.groupby("Temporada")["PuntosPorPartido"].mean(),
    "Tiros al arco (L − V)": diferencia.groupby("Temporada")["TirosArcoPorPartido"].mean(),
    "% equipos con más puntos de local":
        (diferencia["PuntosPorPartido"] > 0).groupby("Temporada").mean() * 100,
})
por_temporada_lv.round(2)

**La ventaja de local desaparece en 2020/21:** los equipos sumaron 1,36 puntos por partido de local y 1,43 de visitante, y solo 9 de los 20 (45 %) rindieron mejor en casa. En 2022/23 y 2023/24, en cambio, 19 de 20 equipos rindieron mejor de local.

Aun así, en 2020/21 los locales siguieron tirando más al arco (+0,46 por partido), aunque la diferencia fue menor que en otras temporadas. Que sigan tirando más pero ya no ganen más puntos es otra señal de que **tirar más no garantiza sumar más**: la relación entre las dos cosas es estadística, no automática.

## 5. Correlaciones

El **coeficiente de correlación de Pearson** (*r*) mide si dos variables suben y bajan juntas, en una escala de −1 a 1:

- Cerca de **1**: cuando una sube, la otra también.
- Cerca de **−1**: cuando una sube, la otra baja.
- Cerca de **0**: no hay una relación lineal.

Como guía orientativa (no es una regla exacta): |*r*| < 0,3 es una relación débil, entre 0,3 y 0,7 moderada y > 0,7 fuerte.

**Qué comparamos:** cada estadística de juego por partido con los puntos por partido y con los goles a favor por partido, **en el mismo nivel** (equipo-temporada), por separado para total, local y visitante. Cada correlación usa **100 observaciones**.

**Valores faltantes:** no hay. Si los hubiera, `np.corrcoef` devolvería `nan`; por eso la función lo comprueba antes de calcular.

In [ ]:
def correlacion(x, y):
    """Calcula la correlación de Pearson entre dos variables con NumPy.

    Args:
        x, y: series con el mismo número de observaciones y sin valores faltantes.

    Returns:
        Coeficiente r, entre -1 y 1.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) != len(y):
        raise ValueError("Las dos variables deben tener el mismo número de observaciones.")
    if np.isnan(x).any() or np.isnan(y).any():
        raise ValueError("Hay valores faltantes.")
    # np.corrcoef devuelve una matriz 2x2; la posición [0, 1] es la correlación entre x e y.
    return np.corrcoef(x, y)[0, 1]


def fuerza(r):
    """Clasifica una correlación según la guía orientativa."""
    if abs(r) < 0.3:
        return "débil"
    elif abs(r) <= 0.7:
        return "moderada"
    else:
        return "fuerte"


# Comprobación: nuestra función da lo mismo que pandas.
assert np.isclose(correlacion(total["TirosArcoPorPartido"], total["PuntosPorPartido"]),
                  total["TirosArcoPorPartido"].corr(total["PuntosPorPartido"]))
print("La función coincide con pandas.")

In [ ]:
tablas_condicion = {"Total": total, "Local": local, "Visitante": visitante}

filas = []
for condicion, tabla in tablas_condicion.items():
    for columna, nombre in VARIABLES_JUEGO.items():
        r_puntos = correlacion(tabla[columna], tabla["PuntosPorPartido"])
        r_goles = correlacion(tabla[columna], tabla["GolesFavorPorPartido"])
        filas.append({
            "Condicion": condicion, "Variable": nombre, "n": len(tabla),
            "r con puntos": r_puntos, "Fuerza": fuerza(r_puntos),
            "r con goles a favor": r_goles,
        })

correlaciones = pd.DataFrame(filas)
correlaciones[correlaciones["Condicion"] == "Total"].round(2)

**Resultados con los 38 partidos (n = 100):**

- **Tiros al arco** es la estadística de juego más relacionada con los puntos (*r* = 0,86, fuerte), seguida de los **tiros** (0,81) y los **córners** (0,72). Las tres se relacionan todavía más con los goles a favor (0,90 los tiros al arco).
- **Faltas** (−0,37) y **amarillas** (−0,33) tienen una relación **negativa moderada**: los equipos con más faltas y tarjetas tienden a sumar menos puntos.
- **Rojas** tiene una relación negativa débil (−0,19).

**Una referencia para comparar:** los goles a favor se relacionan con los puntos con *r* = 0,89, y la diferencia de goles con *r* = 0,97. Es lógico: los puntos se calculan directamente a partir de los goles. Por eso los goles son la referencia y no el foco de la pregunta.

In [ ]:
r_goles_puntos = correlacion(total["GolesFavorPorPartido"], total["PuntosPorPartido"])
r_dg_puntos = correlacion(total["DG"], total["Puntos"])
print(f"Goles a favor ~ puntos: r = {r_goles_puntos:.2f}")
print(f"Diferencia de goles ~ puntos: r = {r_dg_puntos:.2f}")

### ¿Cambia la relación entre local y visitante?

In [ ]:
comparacion_r = correlaciones.pivot(index="Variable", columns="Condicion", values="r con puntos")
comparacion_r = comparacion_r[["Total", "Local", "Visitante"]].loc[list(VARIABLES_JUEGO.values())]
comparacion_r["Local − Visitante"] = comparacion_r["Local"] - comparacion_r["Visitante"]
comparacion_r.round(2)

Las diferencias entre local y visitante son pequeñas: por ejemplo, 0,82 frente a 0,78 en los tiros al arco. Para saber si son reales o simple variación, calculamos las mismas correlaciones **en cada temporada por separado** (20 equipos por temporada). Si la diferencia entre local y visitante fuera real, debería repetirse en todas o casi todas las temporadas.

In [ ]:
filas = []
for temporada in sorted(total["Temporada"].unique()):
    for condicion in ["Local", "Visitante"]:
        tabla = tablas_condicion[condicion]
        tabla = tabla[tabla["Temporada"] == temporada]
        fila = {"Temporada": temporada, "Condicion": condicion, "n": len(tabla)}
        for columna, nombre in VARIABLES_JUEGO.items():
            fila[nombre] = correlacion(tabla[columna], tabla["PuntosPorPartido"])
        filas.append(fila)

por_temporada_r = pd.DataFrame(filas).set_index(["Temporada", "Condicion"])
por_temporada_r.round(2)

In [ ]:
# ¿En cuántas temporadas la correlación es más fuerte (en valor absoluto) de local?
r_local = por_temporada_r.xs("Local", level="Condicion")
r_visitante = por_temporada_r.xs("Visitante", level="Condicion")
mas_fuerte_local = (r_local[list(VARIABLES_JUEGO.values())].abs()
                    > r_visitante[list(VARIABLES_JUEGO.values())].abs()).sum()
rango = (por_temporada_r[list(VARIABLES_JUEGO.values())].max()
         - por_temporada_r[list(VARIABLES_JUEGO.values())].min())

print(pd.DataFrame({
    "Temporadas (de 5) con r más fuerte de local": mas_fuerte_local,
    "Rango de r (10 valores: 5 temporadas × 2 condiciones)": rango.round(2),
}))

**Interpretación:**

- **Tiros al arco:** la correlación con los puntos es fuerte de local y de visitante en todas las temporadas (entre 0,73 y 0,88). A veces es más fuerte de local y a veces de visitante, y la diferencia con las cinco temporadas juntas (0,04) es menor que lo que varía de una temporada a otra. **No hay evidencia de que la relación cambie según la condición.** Con los tiros pasa lo mismo.
- **Faltas:** la relación negativa es algo más fuerte de visitante en 4 de las 5 temporadas (−0,37 frente a −0,29 con todo junto). Es la única diferencia que se repite, pero es pequeña y entre relaciones débiles o moderadas: se puede mencionar como una tendencia, no como una conclusión.
- **Amarillas:** son negativas de local y de visitante, pero cuál es más fuerte cambia según la temporada.
- **Rojas:** es una relación débil y su signo cambia según la temporada y la condición, como se esperaba con tan pocas tarjetas.

**Respuesta a la pregunta guía con estos datos:** lo que distingue a local y visitante es el **nivel** de las estadísticas (de local se tira más y se suman más puntos), no la **fuerza de la relación** entre el juego y los puntos.

### 5.1 Matriz de correlación entre las estadísticas

Hasta aquí comparamos cada estadística con los puntos. Ahora calculamos la correlación **de todas las parejas** a la vez: es la **matriz de correlación**, la misma que muestra la sección *Relaciones y correlaciones* del dashboard.

Usamos las 100 filas de `total` y solo promedios por partido, para no mezclar totales con valores de un partido. Agregamos tres columnas que faltan: victorias y derrotas por partido, y tarjetas (amarillas + rojas) por partido.

In [ ]:
total["VictoriasPorPartido"] = total["V"] / total["PJ"]
total["DerrotasPorPartido"] = total["D"] / total["PJ"]
total["TarjetasPorPartido"] = total["AmarillasPorPartido"] + total["RojasPorPartido"]

VARIABLES_MATRIZ = {
    "GolesFavorPorPartido": "Goles anotados",
    "GolesContraPorPartido": "Goles recibidos",
    "PuntosPorPartido": "Puntos",
    "DerrotasPorPartido": "Derrotas",
    "TirosPorPartido": "Tiros",
    "TirosArcoPorPartido": "Tiros al arco",
    "CornersPorPartido": "Córners",
    "FaltasPorPartido": "Faltas",
    "TarjetasPorPartido": "Tarjetas",
}

# Dos ciclos: para cada pareja de variables calculamos r con nuestra función correlacion().
nombres = list(VARIABLES_MATRIZ.values())
matriz = pd.DataFrame(index=nombres, columns=nombres, dtype=float)
for columna_fila, nombre_fila in VARIABLES_MATRIZ.items():
    for columna_col, nombre_col in VARIABLES_MATRIZ.items():
        matriz.loc[nombre_fila, nombre_col] = correlacion(total[columna_fila], total[columna_col])

# Comprobación: pandas calcula lo mismo con .corr().
assert np.allclose(matriz.values, total[list(VARIABLES_MATRIZ)].corr().values)
matriz.round(2)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Naranja = relación negativa, gris claro = sin relación, morado = relación positiva.
colores = LinearSegmentedColormap.from_list("divergente", ["#eb6834", "#f1eff4", "#5b21b6"])

fig, ax = plt.subplots(figsize=(8, 7))
imagen = ax.imshow(matriz.values.astype(float), cmap=colores, vmin=-1, vmax=1)
ax.set_xticks(range(len(nombres)))
ax.set_xticklabels(nombres, rotation=45, ha="right")
ax.set_yticks(range(len(nombres)))
ax.set_yticklabels(nombres)
for i in range(len(nombres)):
    for j in range(len(nombres)):
        valor = matriz.iloc[i, j]
        ax.text(j, i, f"{valor:.2f}".replace(".", ","), ha="center", va="center", fontsize=9,
                color="white" if abs(valor) >= 0.6 else "#1f1235")
fig.colorbar(imagen, ax=ax, label="r de Pearson")
ax.set_title("Matriz de correlación (100 equipo-temporada, por partido)")
fig.tight_layout()
plt.show()

**Cómo leer la matriz** (cifras con los datos auditados):

- **Tiros al arco y goles anotados** tienen la relación más fuerte entre una estadística de juego y los goles (r = 0,90), por encima de los tiros (0,83) y los córners (0,73).
- **Goles anotados y puntos** (0,89) y **goles recibidos y derrotas** (0,85) son relaciones fuertes y esperables: el resultado de cada partido sale de los goles.
- **Faltas y tarjetas** tienen una relación moderada (0,54): los equipos que hacen más faltas suelen recibir más tarjetas, pero no siempre.
- Las faltas y las tarjetas se relacionan de forma **negativa y débil o moderada** con los puntos (−0,37 y −0,34).

Como siempre, **correlación no es causalidad**: la matriz dice qué estadísticas suben y bajan juntas, no por qué.

### 5.2 ¿Gana siempre el que más tira?

Las correlaciones de arriba usan temporadas completas. Partido a partido la historia es distinta. Unimos cada fila de `equipo_partido` con la de su rival (mismo `PartidoID`) para comparar los tiros de los dos equipos. Es lo que hace el **Detector de sorpresas** del dashboard.

In [ ]:
rival = equipo_partido[["PartidoID", "Equipo", "Tiros", "TirosArco"]].rename(
    columns={"Equipo": "Rival", "Tiros": "TirosRival", "TirosArco": "TirosArcoRival"})
con_rival = equipo_partido.merge(rival, on=["PartidoID", "Rival"])

# El partido sin dato de tiros (Newcastle 2-4 West Ham) queda fuera.
con_rival = con_rival.dropna(subset=["Tiros", "TirosRival"])

mas_tiros = con_rival[con_rival["Tiros"] > con_rival["TirosRival"]]
mas_arco = con_rival[con_rival["TirosArco"] > con_rival["TirosArcoRival"]]

resumen_tiros = pd.DataFrame({
    "Más tiros que el rival": mas_tiros["Resultado"].value_counts(),
    "Más tiros al arco que el rival": mas_arco["Resultado"].value_counts(),
}).rename(index={"V": "Ganó", "E": "Empató", "D": "Perdió"}).loc[["Ganó", "Empató", "Perdió"]]
porcentajes = (100 * resumen_tiros / resumen_tiros.sum()).round(1)
print("Partidos:")
print(resumen_tiros)
print("\nPorcentaje:")
print(porcentajes)

**Resultado:** en los 1.826 casos en que un equipo hizo más tiros que su rival, ganó el **52,4 %** de las veces, empató el 22,3 % y **perdió el 25,3 %** (462 partidos). Con más tiros al arco gana más a menudo (62,8 %) y pierde menos (15,4 %).

Esto encaja con las correlaciones: los tiros al arco se relacionan más con el resultado que los tiros. Pero también muestra un límite del análisis por temporadas: **en un partido concreto, tener más tiros no garantiza ganar**. Por eso el detector de sorpresas solo describe qué estadísticas fueron mayores, sin decir que un equipo "fue mejor".

## 6. Gráficos exploratorios

Cada gráfico se relaciona con uno del dashboard o con un hallazgo de este notebook.

### 6.1 Distribución de los puntos

Antes de comparar, vemos cómo se reparten los puntos totales de los 100 equipo-temporada.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(total["Puntos"], bins=range(0, 105, 5), color=COLOR_LOCAL, edgecolor="white", linewidth=2)
ax.axvline(total["Puntos"].mean(), color=COLOR_TEXTO, linestyle="--", linewidth=1.5)
ax.text(total["Puntos"].mean() + 1, ax.get_ylim()[1] * 0.92,
        f"Media: {total['Puntos'].mean():.1f}", color=COLOR_TEXTO)
ax.set_title("Puntos por temporada de los 100 equipo-temporada (2020/21 a 2024/25)")
ax.set_xlabel("Puntos en la temporada (38 partidos)")
ax.set_ylabel("Número de equipo-temporada")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

Los puntos están muy repartidos: 24 equipo-temporada sumaron menos de 40, 42 entre 40 y 60, 25 entre 61 y 80, y solo 9 más de 80. Esos pocos equipos con más de 80 puntos explican que la media (52,7) quede algo por encima de la mediana (51,5).

### 6.2 Puntos por equipo en una temporada (gráfico 1 del dashboard)

En el dashboard, este gráfico mostrará la temporada que elija el usuario. Aquí usamos 2024/25 como ejemplo.

In [ ]:
TEMPORADA_EJEMPLO = "2024/25"

tabla = total[total["Temporada"] == TEMPORADA_EJEMPLO].sort_values("Puntos")
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(tabla["Equipo"], tabla["Puntos"], color=COLOR_LOCAL, height=0.7)
for posicion, puntos in enumerate(tabla["Puntos"]):
    ax.text(puntos + 0.8, posicion, str(puntos), va="center", fontsize=8, color=COLOR_TEXTO)
ax.set_title(f"Puntos por equipo en {TEMPORADA_EJEMPLO} (calculados con los resultados)")
ax.set_xlabel("Puntos")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### 6.3 Puntos acumulados por número de partido (gráfico 2 del dashboard)

Muestra cómo fue sumando puntos cada equipo a lo largo de la temporada. El eje horizontal es el **número de partido del equipo** (1 a 38), no la jornada oficial.

In [ ]:
EQUIPOS_EJEMPLO = ["Liverpool", "Arsenal", "Man City"]
colores = [COLOR_LOCAL, COLOR_VISITANTE, COLOR_TERCERO]

fig, ax = plt.subplots(figsize=(8, 5.5))
for equipo, color in zip(EQUIPOS_EJEMPLO, colores):
    filtro = (equipo_partido["Temporada"] == TEMPORADA_EJEMPLO) & (equipo_partido["Equipo"] == equipo)
    datos_equipo = equipo_partido[filtro].sort_values("NumeroPartido")
    ax.plot(datos_equipo["NumeroPartido"], datos_equipo["PuntosAcumulados"], color=color, label=equipo)
    # Etiqueta directa al final de la línea.
    ultimo = datos_equipo.iloc[-1]
    ax.text(38.6, ultimo["PuntosAcumulados"], f"{equipo} ({ultimo['PuntosAcumulados']})",
            va="center", fontsize=9, color=COLOR_TEXTO)
ax.set_xlim(1, 38)
ax.set_title(f"Puntos acumulados en {TEMPORADA_EJEMPLO}")
ax.set_xlabel("Número de partido del equipo (orden por fecha)")
ax.set_ylabel("Puntos acumulados")
ax.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

### 6.4 Tiros al arco frente a puntos, de local y de visitante (gráfico 3 del dashboard)

Cada punto es un equipo-temporada. Usamos dos paneles con los mismos ejes para comparar local y visitante sin que los puntos se mezclen.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
paneles = [("Local", local, COLOR_LOCAL), ("Visitante", visitante, COLOR_VISITANTE)]

for ax, (condicion, tabla, color) in zip(ejes, paneles):
    x = tabla["TirosArcoPorPartido"]
    y = tabla["PuntosPorPartido"]
    ax.scatter(x, y, s=36, color=color, alpha=0.8, edgecolor="white", linewidth=1)
    # Recta de tendencia (mínimos cuadrados) con np.polyfit.
    pendiente, intercepto = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, pendiente * xs + intercepto, color=COLOR_TEXTO, linewidth=1.5)
    ax.set_title(f"{condicion}: r = {correlacion(x, y):.2f} (n = {len(tabla)})")
    ax.set_xlabel("Tiros al arco por partido")

ejes[0].set_ylabel("Puntos por partido")
fig.suptitle("Los equipos que tiran más al arco suman más puntos, de local y de visitante",
             fontweight="bold")
plt.tight_layout()
plt.show()

Las dos nubes de puntos tienen una forma parecida y una pendiente parecida. La de visitante está desplazada hacia abajo y a la izquierda: fuera de casa los equipos tiran menos y suman menos, pero la relación entre las dos cosas se mantiene.

`np.polyfit(x, y, 1)` calcula la recta que mejor se ajusta a los puntos. Solo sirve para ver la tendencia; no es un modelo para predecir.

### 6.5 Fuerza de la relación con los puntos, de local y de visitante

Este gráfico resume la respuesta a la pregunta guía: la correlación de cada estadística de juego con los puntos.

In [ ]:
etiquetas = list(VARIABLES_JUEGO.values())
posiciones = np.arange(len(etiquetas))
ancho = 0.38

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(posiciones - ancho / 2 - 0.01, comparacion_r["Local"], ancho, color=COLOR_LOCAL, label="Local")
ax.bar(posiciones + ancho / 2 + 0.01, comparacion_r["Visitante"], ancho, color=COLOR_VISITANTE, label="Visitante")
ax.axhline(0, color=COLOR_TEXTO, linewidth=1)
for posicion, (r_l, r_v) in enumerate(zip(comparacion_r["Local"], comparacion_r["Visitante"])):
    for desplazamiento, r in [(-ancho / 2, r_l), (ancho / 2, r_v)]:
        va = "bottom" if r >= 0 else "top"
        ax.text(posicion + desplazamiento, r + (0.02 if r >= 0 else -0.02), f"{r:.2f}",
                ha="center", va=va, fontsize=8, color=COLOR_TEXTO)
ax.set_xticks(posiciones, etiquetas)
ax.set_ylim(-0.6, 1.0)
ax.set_ylabel("Correlación (r) con puntos por partido")
ax.set_title("Correlación de cada estadística de juego con los puntos (n = 100 por barra)")
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

### 6.6 Ventaja de local por temporada

In [ ]:
temporadas = list(por_temporada_lv.index)
posiciones = np.arange(len(temporadas))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(posiciones - ancho / 2 - 0.01, por_temporada_lv["Puntos local"], ancho, color=COLOR_LOCAL, label="Local")
ax.bar(posiciones + ancho / 2 + 0.01, por_temporada_lv["Puntos visitante"], ancho,
       color=COLOR_VISITANTE, label="Visitante")
for posicion, (p_l, p_v) in enumerate(zip(por_temporada_lv["Puntos local"], por_temporada_lv["Puntos visitante"])):
    ax.text(posicion - ancho / 2, p_l + 0.02, f"{p_l:.2f}", ha="center", fontsize=8, color=COLOR_TEXTO)
    ax.text(posicion + ancho / 2, p_v + 0.02, f"{p_v:.2f}", ha="center", fontsize=8, color=COLOR_TEXTO)
ax.set_xticks(posiciones, temporadas)
ax.set_ylim(0, 1.9)
ax.set_ylabel("Puntos por partido (media de los 20 equipos)")
ax.set_title("Puntos por partido de local y de visitante en cada temporada")
ax.annotate("2020/21: casi toda la temporada\nsin público", xy=(ancho / 2, 1.50), xytext=(0.45, 1.72),
            fontsize=8, color=COLOR_TEXTO, arrowprops={"arrowstyle": "-", "color": COLOR_TEXTO})
ax.legend(frameon=False, loc="upper right")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

## 7. Hallazgos y limitaciones

### Hallazgos (respaldados por los cálculos de este notebook)

1. **Los tiros al arco son la estadística de juego más relacionada con los puntos** (*r* = 0,86, n = 100), seguidos de los tiros (0,81) y los córners (0,72). Faltas (−0,37) y amarillas (−0,33) tienen una relación negativa moderada, y las rojas una débil (−0,19).
2. **La fuerza de esas relaciones es parecida de local y de visitante.** Por ejemplo, 0,82 y 0,78 en los tiros al arco. La diferencia es menor que la variación entre temporadas, así que no podemos afirmar que la relación cambie según la condición. La única diferencia que se repite es pequeña: la relación negativa de las faltas es algo más fuerte de visitante en 4 de las 5 temporadas.
3. **Lo que sí cambia es el nivel.** De local los equipos tiran más (+2,2 tiros y +0,7 tiros al arco por partido), sacan más córners, cometen algo menos de faltas y suman 0,28 puntos más por partido. 73 de los 100 equipo-temporada rindieron mejor en casa.
4. **La excepción es 2020/21**, jugada casi entera sin público: los visitantes ganaron más partidos que los locales (153 frente a 144) y la ventaja de local en puntos desapareció. Es una coincidencia que vale la pena contar, pero no demuestra que la causa sea la falta de público.

### Limitaciones

- **Correlación no es causalidad.** Que los equipos con más tiros al arco sumen más puntos no significa que tirar más *cause* más puntos. Lo más probable es que un factor común, la calidad del equipo, explique las dos cosas.
- **El sentido de la relación puede ser el contrario.** Un equipo que va ganando puede tirar menos porque defiende el resultado. Con datos por temporada no podemos ver ese efecto.
- **Las observaciones no son independientes.** Los mismos equipos aparecen en varias temporadas, y sus datos de local y de visitante salen del mismo equipo. Por eso hablamos de resultados descriptivos y no hacemos pruebas de significación estadística.
- **Las correlaciones por temporada usan solo 20 equipos**, así que varían bastante de una temporada a otra.
- **Tiros al arco y goles están ligados por definición**: casi todo gol es un tiro al arco. Los autogoles son la excepción y el dataset no los identifica.
- **Faltan variables importantes**, como la posesión o la calidad de las ocasiones (goles esperados).
- **Los puntos son los ganados en el campo.** No incluyen las sanciones de Everton (−8) y Nottingham Forest (−4) en 2023/24.
- **Al West Ham 2021/22 le faltan los tiros de un partido**, porque el dato original era imposible.

## 8. Guardar los resultados

Guardamos las tablas y un resumen con las cifras clave. Las cifras del resumen se toman directamente de los cálculos, no se escriben a mano, para que el informe y la presentación usen exactamente los mismos números.

In [ ]:
descriptivas.round(4).to_csv("descriptivas.csv")
correlaciones.round(4).to_csv("correlaciones.csv", index=False)
por_temporada_r.round(4).to_csv("correlaciones_por_temporada.csv")
comparacion.round(4).to_csv("local_vs_visitante.csv")

fila_arco = correlaciones[(correlaciones["Condicion"] == "Total")
                          & (correlaciones["Variable"] == "Tiros al arco")].iloc[0]
resumen = {
    "observaciones_por_condicion": int(len(total)),
    "r_tiros_arco_puntos": {c: round(float(comparacion_r.loc["Tiros al arco", c]), 3)
                            for c in ["Total", "Local", "Visitante"]},
    "r_con_puntos_total": {v: round(float(comparacion_r.loc[v, "Total"]), 3)
                           for v in VARIABLES_JUEGO.values()},
    "r_goles_favor_puntos": round(float(r_goles_puntos), 3),
    "ventaja_local_puntos_por_partido": round(float(diferencia["PuntosPorPartido"].mean()), 3),
    "equipos_temporada_mas_puntos_local": mas_puntos_local,
    "resultados_2020_21": resultados.loc["2020/21"].astype(int).to_dict(),
}
with open("resumen_analisis.json", "w", encoding="utf-8") as archivo:
    json.dump(resumen, archivo, ensure_ascii=False, indent=2)

assert fila_arco["n"] == 100
print(json.dumps(resumen, ensure_ascii=False, indent=2))

## Para terminar

Descarguen los cinco archivos generados: `descriptivas.csv`, `correlaciones.csv`, `correlaciones_por_temporada.csv`, `local_vs_visitante.csv` y `resumen_analisis.json`. El informe y la presentación tomarán las cifras de ahí.

**Fuentes citadas en este notebook**

- Football-Data.co.uk / DataHub. *English Premier League* [Conjunto de datos]. https://datahub.io/football/english-premier-league
- Wikipedia. *2020–21 Premier League*. https://en.wikipedia.org/wiki/2020%E2%80%9321_Premier_League
- Cyprus Mail (5 de mayo de 2021). *Home fans only for final two Premier League matches*. https://cyprus-mail.com/2021/05/05/home-fans-only-for-final-two-premier-league-matches

**Preguntas que deben poder responder en la sustentación**

- ¿Por qué usamos promedios por partido y no totales?
- ¿Qué aporta la mediana que no aporta la media?
- ¿Qué hace `ddof=1` y por qué lo usamos?
- ¿Por qué cada correlación tiene 100 observaciones? ¿Qué es una observación?
- ¿Por qué los goles no son el foco de la pregunta, si son lo más relacionado con los puntos?
- Si los tiros al arco tienen *r* = 0,86 con los puntos, ¿podemos decir que tirar más al arco hace ganar más puntos? ¿Por qué?
- ¿Por qué calculamos las correlaciones por temporada?
- ¿Qué pasó en 2020/21 y por qué no podemos afirmar que se deba a la falta de público?